## Part 3: Evaluation

In [47]:
# imports

from pathlib import Path
import json
import torch
import torch.nn as nn
import sentencepiece as spm
import pandas as pd

DATA_DIR = Path("/srv/data/lt2326-h26/a1")

PROJECT_DIR = Path.cwd().parent
TOKENIZER_DIR = PROJECT_DIR / "artifacts" / "tokenizers"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A30


In [49]:
## Re-loading character tokenizer

with open(TOKENIZER_DIR / "char_vocab.json", encoding="utf-8") as f:
    char_vocab = json.load(f)

char_to_id = {char: i for i, char in enumerate(char_vocab)}

id_to_char = {i: char for i, char in enumerate(char_vocab)}

## Re-creating character encoder

def encode(text):
    token_ids = []

    for char in text:
        if char in char_to_id:
            token_id = char_to_id[char]
        else:
            token_id = char_to_id.get("<unk>", 0) # fallback for <unk>
        token_ids.append(token_id)

    return token_ids

print("Character vocab:", len(char_vocab))
print("UNK:", char_to_id["<unk>"])

Character vocab: 9443
UNK: 0


In [50]:
## Re-loading the BPE tokenizers

small_bpe = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "small_bpe.model")
)

large_bpe = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "large_bpe.model")
)

print("Character:", len(char_vocab))
print("Small BPE:", small_bpe.vocab_size())
print("Large BPE:", large_bpe.vocab_size())

Character: 9443
Small BPE: 3000
Large BPE: 10000


In [51]:
## Encoding wrappers

def encode_char(text):
    return encode(text)

def encode_small_bpe(text):
    return small_bpe.encode(text, out_type=int)

def encode_large_bpe(text):
    return large_bpe.encode(text, out_type=int)

In [52]:
# adding eos to be consistent with training

# original vocab sizes
char_eos_id = len(char_vocab)
small_eos_id = small_bpe.vocab_size()
large_eos_id = large_bpe.vocab_size()

# vocab sizes with eos
char_lm_vocab_size = len(char_vocab) + 1
small_lm_vocab_size = small_bpe.vocab_size() + 1
large_lm_vocab_size = large_bpe.vocab_size() + 1

print("Character:", char_lm_vocab_size, "EOS:", char_eos_id)
print("BPE-3k:", small_lm_vocab_size, "EOS:", small_eos_id)
print("BPE-10k:", large_lm_vocab_size, "EOS:", large_eos_id)

Character: 9444 EOS: 9443
BPE-3k: 3001 EOS: 3000
BPE-10k: 10001 EOS: 10000


In [53]:
# tokenizer conditions
tokenizers = {
    "Character": {
        "encode_fn": encode_char,
        "eos_id": char_eos_id,
        "vocab_size": char_lm_vocab_size,
    },
    "BPE-3k": {
        "encode_fn": encode_small_bpe,
        "eos_id": small_eos_id,
        "vocab_size": small_lm_vocab_size,
    },
    "BPE-10k": {
        "encode_fn": encode_large_bpe,
        "eos_id": large_eos_id,
        "vocab_size": large_lm_vocab_size,
    },
}

In [54]:
## test data streams

# encoding all the files
def build_test_token_stream(file_paths, encode_fn, eos_id):
    token_ids = []

    for path in file_paths:
        with open(path, encoding="utf-8") as f:
            for line in f:
                sentence = line.rstrip("\n")
                encoded_ids = encode_fn(sentence)

                token_ids.extend(encoded_ids)
                token_ids.append(eos_id)

    return torch.tensor(token_ids, dtype=torch.long)

test_paths = [
    DATA_DIR / "test" / "en.txt",
    DATA_DIR / "test" / "tr.txt",
    DATA_DIR / "test" / "zh.txt",
]

# building the streams
languages = ["en", "tr", "zh"]

test_streams = {}

for name, config in tokenizers.items():
    test_streams[name] = {}

    for language in languages: # separate language evaluation this time
        test_path = DATA_DIR / "test" / f"{language}.txt"

        test_streams[name][language] = build_test_token_stream(
            file_paths=[test_path],
            encode_fn=config["encode_fn"],
            eos_id=config["eos_id"],
        )

# sanity checking
for name in tokenizers:
    for language in languages:
        stream = test_streams[name][language]

        print(
            name,
            language,
            "tokens:", len(stream),
            "min:", stream.min().item(),
            "max:", stream.max().item(),
        )

Character en tokens: 372781 min: 0 max: 9443
Character tr tokens: 372134 min: 0 max: 9443
Character zh tokens: 377695 min: 0 max: 9443
BPE-3k en tokens: 230905 min: 0 max: 3000
BPE-3k tr tokens: 249630 min: 0 max: 3000
BPE-3k zh tokens: 367875 min: 0 max: 3000
BPE-10k en tokens: 117765 min: 0 max: 10000
BPE-10k tr tokens: 116374 min: 0 max: 10000
BPE-10k zh tokens: 295479 min: 0 max: 10000


In [55]:
## helper function to count characters for BPC: counting each Unicode character in the original text

def count_characters(path):
    total = 0

    with open(path, encoding="utf-8") as f:
        for line in f:
            sentence = line.rstrip("\n")
            total += len(sentence)

    return total

# checking character counts for each stream + token counts for
test_character_counts = {}

for language in languages:
    test_path = DATA_DIR / "test" / f"{language}.txt"
    test_character_counts[language] = count_characters(test_path)

print("Characters:")
for language in languages:
    print(language, test_character_counts[language])

print("\nToken counts:")
for tokenizer_name in tokenizers:
    for language in languages:
        print(
            tokenizer_name,
            language,
            len(test_streams[tokenizer_name][language])
        )

Characters:
en 368309
tr 368342
zh 368265

Token counts:
Character en 372781
Character tr 372134
Character zh 377695
BPE-3k en 230905
BPE-3k tr 249630
BPE-3k zh 367875
BPE-10k en 117765
BPE-10k tr 116374
BPE-10k zh 295479


In [56]:
## Transformer Language Model from Part 2

class TransformerLM(nn.Module):
    def __init__(
            self,
            vocab_size,
            context_length=128,
            d_model=256,
            n_heads=4,
            n_layers=2,
            dim_feedforward=1024,
            dropout=0.1,
    ):
        super().__init__()

        self.context_length = context_length

        # turning each token ID into a d_model-dimensional vector
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )   # embedding matrix for e.g. 3k [3001, 256]

        # giving each position its own learned representation
        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )   # representing position 0 to 127

        # one transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        # stacking n_layers transformer blocks
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
        )

        # converting hidden representations into vocabulary scores
        self.output_layer = nn.Linear(
            d_model,
            vocab_size
        )   # for e.g. 3k [256, 3001] (produces 3001 scores for possible next token at each position)

    def forward(self, x):

        batch_size, seq_length = x.shape

        # token representations
        token_emb = self.token_embedding(x)

        # positions: 0, 1 ..., seq_lenth -1
        positions = torch.arange(
            seq_length,
            device=x.device
        )

        position_emb = self.position_embedding(positions)

        # combininb token identity and position info
        h = token_emb + position_emb

        # preventing attention to future positions (causal mask)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            seq_length,
            device=x.device
        )

        h = self.transformer(
            h,
            mask=causal_mask
        )

        # scores over the vocabulary ?????
        logits = self.output_layer(h)

        return logits

In [57]:
## reloading the trained models

CONTEXT_LENGTH = 128

def load_trained_model(checkpoint_path, vocab_size):
    model = TransformerLM(
        vocab_size=vocab_size,
        context_length=CONTEXT_LENGTH,
        d_model=256,
        n_heads=4,
        n_layers=2,
        dim_feedforward=1024,
        dropout=0.1,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(device)
    model.eval() # set to evaluation mode

    return model

char_model_eval = load_trained_model(
    MODEL_DIR / "char_lm.pt",
    char_lm_vocab_size,
)

bpe3k_model_eval = load_trained_model(
    MODEL_DIR / "bpe_3k_lm.pt",
    small_lm_vocab_size,
)

bpe10k_model_eval = load_trained_model(
    MODEL_DIR / "bpe_10k_lm.pt",
    large_lm_vocab_size,
)

In [58]:
# parameter count

def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )

# sanity checking
print("Character:", count_parameters(char_model_eval))
print("BPE-3k:", count_parameters(bpe3k_model_eval))
print("BPE-10k:", count_parameters(bpe10k_model_eval))

print("Evaluation modes:")
print(char_model_eval.training)
print(bpe3k_model_eval.training)
print(bpe10k_model_eval.training)

Character: 6457060
BPE-3k: 3151801
BPE-10k: 6742801
Evaluation modes:
False
False
False


In [59]:
# calculating BPC

import math

def calculate_bpc(model, token_stream, num_characters, context_length, device):
    model.eval()

    total_nll = 0.0
    total_predictions = 0

    criterion = nn.CrossEntropyLoss(reduction="sum") # sum instead of mean

    with torch.no_grad():
        # needs at least one input token and one target token
        for start in range(0, len(token_stream) - 1, context_length):

            end = min(start + context_length, len(token_stream) - 1)

            # input tokens
            x = token_stream[start:end]

            # same sequence shifted by one token
            y = token_stream[start + 1:end + 1]

            # adding batch dimension
            x = x.unsqueeze(0).to(device)
            y = y.unsqueeze(0).to(device)

            logits = model(x)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            total_nll += loss.item()
            total_predictions += y.numel()

    bpc = total_nll / (num_characters * math.log(2)) # dividing by Unicode characters * ln(2)

    return {
        "bpc": bpc,
        "total_nll": total_nll,
        "predicted_tokens": total_predictions,
    }

In [60]:
eval_models = {
    "Character": char_model_eval,
    "BPE-3k": bpe3k_model_eval,
    "BPE-10k": bpe10k_model_eval,
}

In [61]:
## running all nine BPC evals

bpc_results = []

for model_name, model in eval_models.items():
    for language in languages:

        result = calculate_bpc(
            model=model,
            token_stream=test_streams[model_name][language],
            num_characters=test_character_counts[language],
            context_length=CONTEXT_LENGTH,
            device=device,
        )

        bpc_results.append({
            "model": model_name,
            "language": language,
            "bpc": result["bpc"],
            "total_nll": result["total_nll"],
            "predicted_tokens": result["predicted_tokens"],
        })

        print(
            f"{model_name:10} | {language} | "
            f"BPC: {result['bpc']:.4f}"
        )

Character  | en | BPC: 2.5294
Character  | tr | BPC: 2.5604
Character  | zh | BPC: 7.0388
BPE-3k     | en | BPC: 2.4393
BPE-3k     | tr | BPC: 2.4913
BPE-3k     | zh | BPC: 6.8495
BPE-10k    | en | BPC: 2.2845
BPE-10k    | tr | BPC: 2.3740
BPE-10k    | zh | BPC: 6.9201


In [ ]:
## representing the results in pd

bpc_df = pd.DataFrame(bpc_results)
bpc_df

bpc_table = bpc_df.pivot(
    index="model",
    columns="language",
    values="bpc"
)

bpc_table

language,en,tr,zh
model,,,
BPE-10k,2.284469,2.373976,6.920147
BPE-3k,2.439326,2.491285,6.849506
Character,2.529374,2.560355,7.038824


For English, BPC decreases from character-level to 3k to 10k. Increasing the BPE vocabulary leads to a better character-normalized performance.  
Turkish shows the same pattern.  
For Chinese, it decreases too between the character-level and the 3k tokenizer, but is considerably higher for any tokenizer than for English and Turkish. between 3k and 10k, there is a slight increase again. Therefore, the 3k BPE model leads to the lowest BPC.  

As seen in Part 1, when I created the tokenizers, going from a 3k vocabulary to a 10k one caused English (1.62 -> 3.23) and Turkish (1.5 -> 3.27) to have a big increase in chars/token. Here, we can see that BPC also improved between those two. For Chinese, between the 3k and the 10k tokenizer chars/token also increased, but less so (1.03 -> 1.29). And BPC did not get better, but worse (6.85 -> 6.92).  


- parameter count: 10k chinese has over double parameters than 3k chinese, but 3k still has lower bpc  

- coverage disadvantages for chinese  


WRITE REPORT QUESTIONS !!!!

In [73]:
## calculating overall BPC

total_characters = sum(test_character_counts.values()) # denominator
print(total_characters)

overall_nll_by_model = bpc_df.groupby("model")["total_nll"].sum()
print(overall_nll_by_model)

overall_nll = overall_nll_by_model / (total_characters * math.log(2))
print(overall_nll)

1104916
model
BPE-10k      2.955769e+06
BPE-3k       3.007222e+06
Character    3.096171e+06
Name: total_nll, dtype: float64
model
BPE-10k      3.859364
BPE-3k       3.926546
Character    4.042688
Name: total_nll, dtype: float64


### Who gets the vocabulary?

In [103]:
## counting token frequencies by language

from collections import Counter

train_files = {
    "en": DATA_DIR / "train" / "en.txt",
    "tr": DATA_DIR / "train" / "tr.txt",
    "zh": DATA_DIR / "train" / "zh.txt",
}

def count_token_freq(tokenizer, path):
    counts = Counter()

    with open(path, encoding="utf-8") as f:
        for line in f:
            sentence = line.rstrip("\n")

            token_ids = tokenizer.encode(
                sentence,
                out_type=int
            )

            counts.update(token_ids)

    return counts


In [104]:
# inspecting all three languages

for language in ["en", "tr", "zh"]:
    print(f"\n--- {language} ---")

    for token_id, count in small_usage[language].most_common(10):
        piece = small_bpe.id_to_piece(token_id)

        print(
            f"{piece!r:15} "
            f"{count:,}"
        )


--- en ---
'e'             81,317
'▁'             74,629
't'             58,030
'o'             51,934
's'             51,636
'h'             40,327
'er'            39,682
'.'             38,161
'i'             34,505
'▁the'          34,067

--- tr ---
'▁'             85,320
'i'             67,939
'ı'             66,524
'a'             65,854
'e'             48,219
'm'             44,420
'l'             44,173
'u'             38,546
'd'             38,372
'an'            37,892

--- zh ---
','             111,127
'▁'             76,373
'的'             70,916
'<unk>'         54,480
'。'             52,898
'1'             24,108
'、'             23,175
'在'             22,612
'年'             22,266
'一'             21,134


Primarily associated with a language: 80% of tokens's occurences in that language, else: shared

In [105]:
# helper function to classify tokens

def classify_token(en_count, tr_count, zh_count, threshold=0.8): # 80% threshold for association
    total = en_count + tr_count + zh_count

    if total == 0:
        return "unused"

    en_share = en_count / total
    tr_share = tr_count / total
    zh_share = zh_count / total

    if en_share >= threshold:
        return "English"
    elif tr_share >= threshold:
        return "Turkish"
    elif zh_share >= threshold:
        return "Chinese"
    else:
        return "Shared"

In [106]:
# getting 3k usage counts

small_usage = {}

for language, path in train_files.items():
    small_usage[language] = count_token_freq(
        large_bpe,
        path
    )

# looping over 3k vocab

small_vocab_analysis = []

for token_id in range(small_bpe.vocab_size()):
    en_count = small_usage["en"][token_id]
    tr_count = small_usage["tr"][token_id]
    zh_count = small_usage["zh"][token_id]

    total = en_count + tr_count + zh_count

    if total > 0:
        en_share = en_count / total
        tr_share = tr_count / total
        zh_share = zh_count / total
    else:
        en_share = 0
        tr_share = 0
        zh_share = 0

    category = classify_token(
        en_count,
        tr_count,
        zh_count
    )

    small_vocab_analysis.append({
        "token_id": token_id,
        "token": small_bpe.id_to_piece(token_id),
        "en_count": en_count,
        "tr_count": tr_count,
        "zh_count": zh_count,
        "total_count": total,
        "en_share": en_share,
        "tr_share": tr_share,
        "zh_share": zh_share,
        "category": category,
    })

small_vocab_df = pd.DataFrame(small_vocab_analysis)

small_vocab_df.head()
small_vocab_df["category"].value_counts()
small_vocab_df["category"].value_counts(normalize=True) * 100

category
Turkish    29.433333
Shared     26.666667
Chinese    24.033333
English    19.766667
unused      0.100000
Name: proportion, dtype: float64

In [107]:
# getting 10k usage counts

large_usage = {}

for language, path in train_files.items():
    large_usage[language] = count_token_freq(
        large_bpe,
        path
    )


# looping over 10k vocab

large_vocab_analysis = []

for token_id in range(large_bpe.vocab_size()):
    en_count = large_usage["en"][token_id]
    tr_count = large_usage["tr"][token_id]
    zh_count = large_usage["zh"][token_id]

    total = en_count + tr_count + zh_count

    if total > 0:
        en_share = en_count / total
        tr_share = tr_count / total
        zh_share = zh_count / total
    else:
        en_share = 0
        tr_share = 0
        zh_share = 0

    category = classify_token(
        en_count,
        tr_count,
        zh_count
    )

    large_vocab_analysis.append({
        "token_id": token_id,
        "token": large_bpe.id_to_piece(token_id),
        "en_count": en_count,
        "tr_count": tr_count,
        "zh_count": zh_count,
        "total_count": total,
        "en_share": en_share,
        "tr_share": tr_share,
        "zh_share": zh_share,
        "category": category,
    })

large_vocab_df = pd.DataFrame(large_vocab_analysis)

large_vocab_df.head()
large_vocab_df["category"].value_counts()
large_vocab_df["category"].value_counts(normalize=True) * 100

category
Chinese    51.16
Turkish    18.80
Shared     15.73
English    14.22
unused      0.09
Name: proportion, dtype: float64

In [ ]:
# excluding <unk> ???? RIGHT PLACE ???

small_vocab_regular = small_vocab_df[
    small_vocab_df["token"] != "<unk>"
]

large_vocab_regular = large_vocab_df[
    large_vocab_df["token"] != "<unk>"
]

In [109]:
# comparing 3k and 10k

allocation_3k = small_vocab_df["category"].value_counts()
allocation_10k = large_vocab_df["category"].value_counts()

allocation_df = pd.DataFrame({
    "BPE-3k": allocation_3k,
    "BPE-10k": allocation_10k,
}).fillna(0)

allocation_df

# percentages
allocation_pct_df = pd.DataFrame({
    "BPE-3k": small_vocab_df["category"].value_counts(normalize=True) * 100,
    "BPE-10k": large_vocab_df["category"].value_counts(normalize=True) * 100,
}).fillna(0)

allocation_pct_df

,BPE-3k,BPE-10k
category,,
Chinese,24.033333,51.16
English,19.766667,14.22
Shared,26.666667,15.73
Turkish,29.433333,18.80
unused,0.100000,0.09


In [110]:
# inspecting tokens for each category

small_vocab_df[
    small_vocab_df["category"] == "English"
].sort_values(
    "total_count",
    ascending=False
).head(10)

small_vocab_df[
    small_vocab_df["category"] == "Turkish"
].sort_values(
    "total_count",
    ascending=False
).head(10)

small_vocab_df[
    small_vocab_df["category"] == "Chinese"
].sort_values(
    "total_count",
    ascending=False
).head(10)

small_vocab_df[
    small_vocab_df["category"] == "Shared"
].sort_values(
    "total_count",
    ascending=False
).head(10)

large_vocab_df[
    large_vocab_df["category"] == "English"
].sort_values(
    "total_count",
    ascending=False
).head(10)

large_vocab_df[
    large_vocab_df["category"] == "Turkish"
].sort_values(
    "total_count",
    ascending=False
).head(10)

large_vocab_df[
    large_vocab_df["category"] == "Chinese"
].sort_values(
    "total_count",
    ascending=False
).head(10)

large_vocab_df[
    large_vocab_df["category"] == "Shared"
].sort_values(
    "total_count",
    ascending=False
).head(10)

,token_id,token,en_count,tr_count,zh_count,total_count,en_share,tr_share,zh_share,category
7129,7129,",",20647,16843,101694,139184,0.148343,0.121012,0.730644,Shared
7138,7138,.,33557,30828,2386,66771,0.502568,0.461697,0.035734,Shared
7163,7163,',3319,12049,300,15668,0.211833,0.769020,0.019147,Shared
7161,7161,),1585,1737,7736,11058,0.143335,0.157081,0.699584,Shared
7182,7182,-,3827,1830,3220,8877,0.431114,0.206151,0.362735,Shared
7120,7120,a,2871,5161,763,8795,0.326435,0.586811,0.086754,Shared
3,3,in,2139,4707,415,7261,0.294588,0.648258,0.057155,Shared
7121,7121,i,1888,4676,677,7241,0.260737,0.645767,0.093495,Shared
7119,7119,e,2818,3187,688,6693,0.421037,0.476169,0.102794,Shared
115,115,▁(,3326,2364,915,6605,0.503558,0.357911,0.138531,Shared
